In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays

In [3]:
inferencia = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')
inferencia.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [4]:
inferencia = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')

inferencia_df = inferencia.copy()
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])
inferencia_df = inferencia_df.sort_values(
    ['producto_id', 'fecha']
).copy()
inferencia_df['_anio'] = inferencia_df['fecha'].dt.year

# Crear rezagos y media móvil sin descartar filas con valores nulos.
for lag in range(1, 8):
    inferencia_df[f'unidades_vendidas_lag_{lag}'] = (
        inferencia_df
        .groupby(['producto_id', '_anio'])['unidades_vendidas']
        .shift(lag)
    )

inferencia_df['media_movil_7_dias'] = (
    inferencia_df
    .groupby(['producto_id', '_anio'])['unidades_vendidas']
    .transform(
        lambda serie: serie.shift(1).rolling(window=7, min_periods=7).mean()
    )
)

inferencia_df = inferencia_df.drop(columns=['_anio'])

# Variables de precio.
inferencia_df['descuento_porcentaje'] = (
    (inferencia_df['precio_venta'] - inferencia_df['precio_base'])
    / inferencia_df['precio_base']
) * 100

columnas_competencia = ['Amazon', 'Decathlon', 'Deporvillage']
inferencia_df['precio_competencia'] = (
    inferencia_df[columnas_competencia].mean(axis=1)
)
inferencia_df['ratio_precio'] = (
    inferencia_df['precio_venta'] / inferencia_df['precio_competencia']
)
inferencia_df = inferencia_df.drop(columns=columnas_competencia)

# Aplicar el mismo one-hot encoding del entrenamiento.
for columna in ['nombre', 'categoria', 'subcategoria']:
    inferencia_df[f'{columna}_h'] = inferencia_df[columna]

inferencia_df = pd.get_dummies(
    inferencia_df,
    columns=['nombre_h', 'categoria_h', 'subcategoria_h'],
    drop_first=False
)

# Alinear nombres y orden de columnas con el dataframe procesado de entrenamiento.
columnas_entrenamiento = pd.read_csv(
    '../data/processed/df.csv',
    nrows=0
).columns.tolist()

columnas_dummies = [
    columna for columna in inferencia_df.columns
    if columna.startswith(('nombre_h_', 'categoria_h_', 'subcategoria_h_'))
]

dummies_desconocidas = sorted(
    set(columnas_dummies) - set(columnas_entrenamiento)
)

if dummies_desconocidas:
    raise ValueError(
        f'Categorías no vistas en entrenamiento: {dummies_desconocidas}'
    )

inferencia_df = inferencia_df.reindex(
    columns=columnas_entrenamiento,
    fill_value=False
)

# Eliminar octubre únicamente, después de completar las transformaciones.
inferencia_df = inferencia_df.loc[
    inferencia_df['fecha'].dt.month != 10
].copy()

if inferencia_df.empty:
    raise ValueError('No quedan registros tras eliminar octubre.')

# Predictoras del modelo final.
predictoras_modelo = [
    'precio_base',
    'es_estrella',
    'precio_venta',
    'descuento_porcentaje',
    *[f'unidades_vendidas_lag_{lag}' for lag in range(1, 8)],
    'media_movil_7_dias',
]

X_inferencia = inferencia_df[predictoras_modelo]

inferencia_df.to_csv(
    '../data/processed/inferencia_df_transformado.csv',
    index=False
)

print('DataFrame guardado en data/processed/inferencia_df_transformado.csv')
print(f'Forma de inferencia_df: {inferencia_df.shape}')
print(f'Forma de X_inferencia: {X_inferencia.shape}')
print(f'Registros de noviembre: {(inferencia_df["fecha"].dt.month == 11).sum()}')
display(inferencia_df)

DataFrame guardado en data/processed/inferencia_df_transformado.csv
Forma de inferencia_df: (720, 65)
Forma de X_inferencia: (720, 12)
Registros de noviembre: 720


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
192,2025-11-02,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
216,2025-11-03,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
240,2025-11-04,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
264,2025-11-05,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
791,2025-11-26,PROD_024,Lotuscrafts Yoga Bolster,Wellness,Cojín Yoga,50,False,NaN,50.85,NaN,...,False,False,False,False,False,False,False,False,False,False
815,2025-11-27,PROD_024,Lotuscrafts Yoga Bolster,Wellness,Cojín Yoga,50,False,NaN,49.82,NaN,...,False,False,False,False,False,False,False,False,False,False
839,2025-11-28,PROD_024,Lotuscrafts Yoga Bolster,Wellness,Cojín Yoga,50,False,NaN,42.50,NaN,...,False,False,False,False,False,False,False,False,False,False
863,2025-11-29,PROD_024,Lotuscrafts Yoga Bolster,Wellness,Cojín Yoga,50,False,NaN,50.37,NaN,...,False,False,False,False,False,False,False,False,False,False


In [6]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'unidades_vendidas_lag_1', 'unidades_vendidas_lag_2',
       'unidades_vendidas_lag_3', 'unidades_vendidas_lag_4',
       'unidades_vendidas_lag_5', 'unidades_vendidas_lag_6',
       'unidades_vendidas_lag_7', 'media_movil_7_dias', 'descuento_porcentaje',
       'precio_competencia', 'ratio_precio',
       'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23',
       'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552',
       'nombre_h_Columbia Silver Ridge',
       'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900',
       'nombre_h_Domyos Kit Mancuernas 20kg',
       'nombre_h_Gaiam Premium Yoga Block', 'nombre_h_Liforme Yoga Pad',
       'nombre_h_Lotuscrafts Yoga Bolster', 'nombre_h_Manduka PRO Yoga Mat',
       'nombre_h_Merrell Moab 2 GTX',
       'nombre_h_New Balance Fresh